# Disease Prediction from Tabular Data using ChatGPT API

In [1]:
!pip install openai pandas matplotlib --quiet

In [2]:
import pandas as pd
import openai
import getpass
import matplotlib.pyplot as plt

In [44]:
# Securely input API key
api_key = getpass.getpass("MY Keys")
client = OpenAI(api_key=api_key)

MY Keys··········


## Load Synthea Dataset (patients.csv, conditions.csv)

In [6]:
# Loading patients table
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("patients.csv")
df.head()

Saving patients.csv to patients.csv


,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,...,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
0,f0f3bc8d-ef38-49ce-a2bd-dfdda982b271,2017-08-24,NaN,999-68-6630,NaN,NaN,NaN,Jacinto644,Kris249,NaN,...,Beverly Massachusetts US,888 Hickle Ferry Suite 38,Springfield,Massachusetts,Hampden County,1106.0,42.151961,-72.598959,8446.49,1499.08
1,067318a4-db8f-447f-8b6e-f2f61e9baaa5,2016-08-01,NaN,999-15-5895,NaN,NaN,NaN,Alva958,Krajcik437,NaN,...,Boston Massachusetts US,1048 Skiles Trailer,Walpole,Massachusetts,Norfolk County,2081.0,42.177370,-71.281353,89893.40,1845.72
2,ae9efba3-ddc4-43f9-a781-f72019388548,1992-06-30,NaN,999-27-3385,S99971451,X53218815X,Mr.,Jayson808,Fadel536,NaN,...,Springfield Massachusetts US,1056 Harris Lane Suite 70,Chicopee,Massachusetts,Hampden County,1020.0,42.181642,-72.608842,577445.86,3528.84
3,199c586f-af16-4091-9998-ee4cfc02ee7a,2004-01-09,NaN,999-73-2461,S99956432,NaN,NaN,Jimmie93,Harris789,NaN,...,Worcester Massachusetts US,201 Mitchell Lodge Unit 67,Pembroke,Massachusetts,Plymouth County,NaN,42.075292,-70.757035,336701.72,2705.64
4,353016ea-a0ff-4154-85bb-1cf8b6cedf20,1996-11-15,NaN,999-60-7372,S99917327,X58903159X,Mr.,Gregorio366,Auer97,NaN,...,Patras Achaea GR,1050 Lindgren Extension Apt 38,Boston,Massachusetts,Suffolk County,2135.0,42.352434,-71.028610,484076.34,3043.04


## Select Features & Create Labels (Hypertension)

In [7]:
df["AGE"] = 2020 - pd.to_datetime(df["BIRTHDATE"]).dt.year
df["SEX"] = df["GENDER"].map({"M": "Male", "F": "Female"})

In [8]:
# Loading conditions table

from google.colab import files
uploaded = files.upload()

conditions = pd.read_csv("conditions.csv")
conditions.head()

Saving conditions.csv to conditions.csv


,START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION
0,2019-02-15,2019-08-01,f0f3bc8d-ef38-49ce-a2bd-dfdda982b271,d5ee30a9-362f-429e-a87a-ee38d999b0a5,65363002,Otitis media
1,2019-10-30,2020-01-30,f0f3bc8d-ef38-49ce-a2bd-dfdda982b271,8bca6d8a-ab80-4cbf-8abb-46654235f227,65363002,Otitis media
2,2020-03-01,2020-03-30,f0f3bc8d-ef38-49ce-a2bd-dfdda982b271,681c380b-3c84-4c55-80a6-db3d9ea12fee,386661006,Fever (finding)
3,2020-03-01,2020-03-01,f0f3bc8d-ef38-49ce-a2bd-dfdda982b271,681c380b-3c84-4c55-80a6-db3d9ea12fee,840544004,Suspected COVID-19
4,2020-03-01,2020-03-30,f0f3bc8d-ef38-49ce-a2bd-dfdda982b271,681c380b-3c84-4c55-80a6-db3d9ea12fee,840539006,COVID-19


In [9]:
# Doing prefiltering for hypertension data
hypertension_patients = conditions[conditions["DESCRIPTION"].str.contains("Hypertension", case=False)]["PATIENT"].unique()

df["HYPERTENSION"] = df["Id"].isin(hypertension_patients)
df = df[["Id", "AGE", "SEX", "HYPERTENSION"]]
df.head()

,Id,AGE,SEX,HYPERTENSION
0,f0f3bc8d-ef38-49ce-a2bd-dfdda982b271,3,Male,False
1,067318a4-db8f-447f-8b6e-f2f61e9baaa5,4,Female,False
2,ae9efba3-ddc4-43f9-a781-f72019388548,28,Male,True
3,199c586f-af16-4091-9998-ee4cfc02ee7a,16,Female,False
4,353016ea-a0ff-4154-85bb-1cf8b6cedf20,24,Male,False


## Generate Prompts for GPT

In [36]:
# Building prompt
def build_prompt(row):
    return f"""Patient Summary:
Age: {row["AGE"]}
Sex: {row["SEX"]}

Task: Based on the age and sex, determine if this patient is likely to have hypertension. Explain your reasoning step by step."""

df["prompt"] = df.apply(build_prompt, axis=1)
df["prompt"].iloc[0]


'Patient Summary:\nAge: 3\nSex: Male\n\nTask: Based on the age and sex, determine if this patient is likely to have hypertension. Explain your reasoning step by step.'

## Call GPT API (Few-shot + Chain of Thought Style)

In [35]:
# Building calling gpt

def call_gpt(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=250
    )
    return response.choices[0].message.content


In [40]:
# Run on 20 data
sample_df = df.sample(20, random_state=42).copy()
sample_df["gpt_response"] = sample_df["prompt"].apply(call_gpt)
sample_df[["prompt", "gpt_response", "HYPERTENSION"]]

,prompt,gpt_response,HYPERTENSION
7636,Patient Summary:\nAge: 24\nSex: Male\n\nTask: ...,To determine if a 24-year-old male is likely t...,False
4640,Patient Summary:\nAge: 26\nSex: Female\n\nTask...,To determine if a 26-year-old female is likely...,True
11160,Patient Summary:\nAge: 62\nSex: Male\n\nTask: ...,To determine the likelihood of hypertension in...,True
12292,Patient Summary:\nAge: 70\nSex: Male\n\nTask: ...,To determine the likelihood of hypertension in...,False
6742,Patient Summary:\nAge: 106\nSex: Female\n\nTas...,To determine the likelihood of hypertension in...,False
6480,Patient Summary:\nAge: 56\nSex: Female\n\nTask...,To determine the likelihood of a 56-year-old f...,False
3684,Patient Summary:\nAge: 51\nSex: Female\n\nTask...,Determining the likelihood of hypertension in ...,False
1493,Patient Summary:\nAge: 88\nSex: Male\n\nTask: ...,To determine if an 88-year-old male is likely ...,False
1213,Patient Summary:\nAge: 68\nSex: Male\n\nTask: ...,To determine if a 68-year-old male is likely t...,False
7824,Patient Summary:\nAge: 54\nSex: Female\n\nTask...,Determining the likelihood of hypertension in ...,False


## Evaluation of GPT Outputs

In [41]:
# Printing response
for _, row in sample_df.iterrows():
    print("="*50)
    print("GPT Prediction:\n", row["gpt_response"])
    print("Actual Hypertension Status:", row["HYPERTENSION"])

GPT Prediction:
 To determine if a 24-year-old male is likely to have hypertension, we need to consider several factors:

1. **Prevalence of Hypertension by Age and Sex:**
   - Hypertension is less common in younger adults compared to older adults. According to health statistics, the prevalence of hypertension increases with age. Young adults, particularly those under 30, generally have a lower risk compared to older populations.
   - Males have a slightly higher prevalence of hypertension compared to females in younger age groups. However, this difference is not very pronounced in individuals in their early 20s.

2. **Risk Factors for Hypertension:**
   - While age and sex are non-modifiable risk factors, there are several other risk factors that can contribute to hypertension, including obesity, physical inactivity, high salt intake, excessive alcohol consumption, smoking, stress, and a family history of hypertension.
   - In a 24-year-old male, if these risk factors are present, the

## Accuracy Evaluation of GPT Predictions

In [45]:
# Evaluating GPT prediction vs actual hypertension labels

def classify_response(response_text):
    response_text = str(response_text).lower()
    if "not likely" in response_text or "unlikely" in response_text or "no hypertension" in response_text:
        return False
    elif "likely" in response_text or "has hypertension" in response_text:
        return True
    else:
        return None  # Ambiguous response

# Classify responses
sample_df["gpt_predicted_label"] = sample_df["gpt_response"].apply(classify_response)

# Filter out unclear predictions
filtered_df = sample_df.dropna(subset=["gpt_predicted_label"])

# Ensuring both columns are boolean
filtered_df["gpt_predicted_label"] = filtered_df["gpt_predicted_label"].astype(bool)
filtered_df["HYPERTENSION"] = filtered_df["HYPERTENSION"].astype(bool)

# Accuracy metrics
from sklearn.metrics import accuracy_score, confusion_matrix

accuracy = accuracy_score(filtered_df["HYPERTENSION"], filtered_df["gpt_predicted_label"])
conf_matrix = confusion_matrix(filtered_df["HYPERTENSION"], filtered_df["gpt_predicted_label"])

print("GPT Accuracy on sample: {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(conf_matrix)



GPT Accuracy on sample: {accuracy * 100:.2f}%
Confusion Matrix:
[[ 0 12]
 [ 0  5]]


<ipython-input-45-89c62d826a0b>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["gpt_predicted_label"] = filtered_df["gpt_predicted_label"].astype(bool)
<ipython-input-45-89c62d826a0b>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["HYPERTENSION"] = filtered_df["HYPERTENSION"].astype(bool)
